# M6 · Combined optimization and the quality gate

**Learning objective:** measure the combined BEST configuration (Turbo distilled 8-step + INT8 weights) against B0 on **your** W7900, produce paired quality evidence, and keep the final verdict human. The published 6.12× is a reference; your number is your own.

Reference lesson: [M06 lesson](../course/M06_combined_quality/LESSON.md) · [中文](./07_combined_and_quality.zh-CN.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse explicitly selected physical runs (ids shown + classified)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="07")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

from lab import student_provenance as prov

def _origin_stamp(category: str) -> str:
    """Stamp text for a provenance category; reuse is NEVER labelled as
    measured-today (fail-closed student provenance contract)."""
    return prov.stamp_text(category)

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse earlier physical runs to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing explicitly selected physical run(s) "
          "instead of launching new ones (metered-GPU mode; provenance shown)")
    return False

## 1 · Run B0 and BEST physically
Both arms with the profile matching your mode; reuse of your M1 B0 run is explicit (shown run_id).

In [ ]:
def run_m6_pair():
    prof = "classroom" if session.mode == "GPU_CLASSROOM" else "research"
    b0 = session.run("B0", profile=prof)
    best = session.run("BEST", profile=prof)
    return b0, best

m6 = gpu_cell(run_m6_pair) if want_own_arms() else None
m6_origin = "CURRENT_SESSION_PHYSICAL"
if m6 is None:
    ids = [session.latest_campaign_run("B0"), session.latest_campaign_run("BEST")]
    if all(ids):
        m6 = (session.consume(ids[0], why="explicit reuse (shown run_id): B0"),
              session.consume(ids[1], why="explicit reuse (shown run_id): BEST"))
        m6_origin = session.consumed[-1].get("category", "COMMITTED_PHYSICAL_REFERENCE")
if m6 is not None:
    ab = rt.ab_compare(m6[0], m6[1], label="B0 vs BEST (turbo+int8)")
    print({k: ab[k] for k in ("classification", "speedup_median",
                              "latency_delta_s", "vram_delta_mb")})
    _stamp(_origin_stamp(m6_origin), f"B0/BEST speedup {ab['speedup_median']}x "
          f"[{ab['classification']}] vs frozen reference 6.12x")
    fig = rt.plot_ab(ab) or rt.plot_ab(ab, metric="vram")
    if fig: fig.show()

## 2 · The quality axis — paired evidence, human verdict
Combined changes (model variant × schedule × precision) get **distance semantics**. Produce or inspect the paired grid + native-resolution poster crops; score with the three-tier rubric. LPIPS here is a perceptual distance that *triages* review — it never issues the verdict.

> **Quality decisions stay human.** No cell auto-approves an image-quality verdict. Historical owner verdicts apply **only** to their originally evaluated deployment scope; a new physical run needs its own review (`PENDING_HUMAN_REVIEW` until then).

In [ ]:
print("three-tier rubric:  (1) no perceptible change  "
      "(2) slight degradation, still deployable  (3) unacceptable")
print()
def run_own_quality():
    session.run_quality("BEST", reference="B0")
    qdir = ROOT / "runs/quality/BEST"
    for f in sorted(qdir.glob("*.png")):
        print("  your artifact:", f.name)
    import json as _j
    lp = qdir / "lpips_table.json"
    if lp.is_file():
        print(_j.dumps(_j.loads(lp.read_text()), indent=1)[:800])
    return qdir

q = gpu_cell(run_own_quality)
if q is None:
    print("frozen campaign artifacts (reference):")
    for f in sorted((ROOT / "evidence/quality/BEST").glob("*")):
        print("  ", f.name)
    print("historical M6 BEST independent human review: PENDING_HUMAN_REVIEW "
          "(owner decision boundary preserved)")
_stamp("GATE STATUS", "your verdict on YOUR images is PENDING_HUMAN_REVIEW "
      "until you complete the rubric on the grid above")

## 3 · Read your own combination honestly
1. Is your measured B0/BEST ratio close to 6.12×? If not, which arm differs from the reference (compare B0 and BEST medians separately against 55.206 s / 9.023 s)?
2. The combined ratio is NOT F1×F2×L1 multiplied — why?
3. BEST's VRAM peak is ~6.6 GB below B0's — what does that enable?

In [ ]:
if m6 is not None:
    b0s, bests = m6[0].stats(), m6[1].stats()
    print(f"B0   {b0s['median_s']:7.2f}s (ref 55.206)  VRAM {b0s['vram_device_peak_mb']:.0f}MB (ref 33153)")
    print(f"BEST {bests['median_s']:7.2f}s (ref 9.023)  VRAM {bests['vram_device_peak_mb']:.0f}MB (ref 26514)")
    print("report the delta per arm, not only the ratio")

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## Acceptance criteria

- [ ] You can state which numbers are **measured on this GPU today**, which are
      **historical reference values**, and which are **estimates/simulations**.
- [ ] Every physical run you produced has a visible `run_id` under `runs/`.
- [ ] You did not present a smoke/classroom population as a formal result.
- [ ] You can explain one limitation of each measurement you reported.

## Reproduce & contribute

```bash
python -m lab doctor                                  # admission evidence -> runs/doctor.json
python -m lab run --experiment BEST --date <your-tag>  # full research protocol
python -m lab report --scope student --kind all       # aggregate YOUR runs only
```

Commit your `runs/<run-id>/*.json` records (never `evidence/`) and open a PR
describing: hypothesis → protocol → numbers → limitations. Negative results
are welcome — a measured regression is a scientific outcome, not a failure.